# Day 2 Pair 7 - Will and Bakul

After a bit of switching around between teams

In [ ]:
import asyncio
import datetime
import os
import sys
import threading
import time
import urllib.request
from collections.abc import Callable
from copy import copy
from dataclasses import dataclass
from http.server import BaseHTTPRequestHandler, HTTPServer
from pathlib import Path

import jwt  # PyJWT — for MCP bonus section
import numpy as np
from control_arena import ControlAgent, control_agent
from control_arena.analysis import samples_df
from control_arena.eval import run_monitor_for_eval_logs
from control_arena.micro_protocol import trusted_monitoring_builder
from control_arena.monitor import (
    MonitorResponse,
    basic_monitor_builder,
    cot_prompt_formatter,
)
from control_arena.monitor._types import MonitorScoreRange
from control_arena.policy import (
    attack_policy_builder,
    honest_policy_builder,
    trusted_policy_builder,
    untrusted_policy_builder,
)
from control_arena.scorers import max_suspicion_scorer
from control_arena.settings.apps.apps_solution_policy import apps_solution_policy
from inspect_ai import score as inspect_score
from inspect_ai.agent import Agent, AgentState
from inspect_ai.log import write_eval_log
from inspect_ai.model import ChatMessageAssistant
from inspect_ai.tool import Tool
from mcp.server.fastmcp import FastMCP  # for MCP bonus section
from openai import OpenAI
from pydantic import Field


import os
import sys
from pathlib import Path

from openai import OpenAI

for _path in [
    str(Path.cwd().parent),
    str(Path.cwd().parent.parent),
]:
    if _path not in sys.path:
        sys.path.insert(0, _path)

from aisb_utils import report
from aisb_utils.env import load_dotenv

load_dotenv()

# Paths relative to this file
SCRIPT_DIR = Path.cwd().parent

# OpenRouter client for exercises 1-2
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY", ""),
)

# A cheap, small model for exercises where we *want* the model to be easily manipulated
SMALL_MODEL = "meta-llama/llama-3-8b-instruct"


Exercise 1.1 mapping the attack surface

Channels through which untrusted input can reach the context window for a coding agent
* Results passed back from MCP server - the response (whether file based, web based etc). If the MCP server isn't doing filtering itself - where the MCP can give some security points - any website contents, we can point to a website that has a prompt in it.
  * e.g. you can do a supply chain attack if you get it to return a malicious package for some specified use case
  * Code injection, manipulation of database queries SQL injection these things
* MCP server is itself an API: the tool specifications it returns could also include untrusted input if e.g. it's compromised.
* User chat input directly - standard direct prompt injection.

Reference solution mentions potential RAG injection - there's no mention in the problem statement about there being a RAG present, but there is a possibility that there is one